# Tracing containerized service dependencies

## Purpose

This pattern combines observability with containerization: three services that each run in their own container (frontend calls api, api calls db), traced end to end so a slow request can be blamed on one dependency edge. The collection shape follows the OpenTelemetry Collector model — receivers take data in, processors transform it, exporters send it on — with lightweight agents next to each container forwarding to one central gateway that applies policy before export (https://opentelemetry.io/docs/collector/architecture/).

Below I generate synthetic spans for one request path, rebuild the container dependency graph from them, find the slowest edge, and sketch the agent-plus-gateway collector config that would carry these spans in a real setup.

In [ ]:
# last_verified: 2026-09-19 · monitoring-observability-concepts (stdlib only)
import json
import random
from collections import defaultdict

## Step 1 — model the containers and their spans

Each service is one container. A request enters at frontend, which calls api, which calls db. Every span carries the trace id, its own id, its parent id, and the container that emitted it.

In [ ]:
SERVICES = {
    "frontend": {"container": "shop-frontend-1", "calls": ["api"]},
    "api": {"container": "shop-api-1", "calls": ["db"]},
    "db": {"container": "shop-db-1", "calls": []},
}


def generate_spans(seed=7):
    """One trace through frontend -> api -> db with per-span durations."""
    rng = random.Random(seed)
    trace_id = "t-%04d" % rng.randrange(10000)
    spans, parent, clock = [], None, 0.0
    for i, svc in enumerate(["frontend", "api", "db"]):
        duration = round(rng.uniform(2, 8) + (25 if svc == "db" else 0), 1)
        span = {"trace_id": trace_id, "span_id": "s-%d" % i,
                "parent_id": parent, "service": svc,
                "container": SERVICES[svc]["container"],
                "operation": "GET /items", "start_ms": round(clock, 1),
                "duration_ms": duration, "status": "ok"}
        spans.append(span)
        parent, clock = span["span_id"], clock + duration
    return spans


spans = generate_spans()
print(json.dumps(spans, indent=2))

## Step 2 — rebuild the dependency graph from the spans

Parent links give the caller--callee edges between containers. The slowest edge is where the request spent most of its time — here it should point at the db container.

In [ ]:
by_id = {s["span_id"]: s for s in spans}
edges = defaultdict(list)
for s in spans:
    if s["parent_id"] is not None:
        edges[by_id[s["parent_id"]]["service"]].append((s["service"], s["duration_ms"]))

print("container dependency edges (caller -> callee, ms):")
for caller, callees in edges.items():
    for callee, ms in callees:
        print("  %s (%s) -> %s (%s): %s ms"
              % (caller, SERVICES[caller]["container"],
                 callee, SERVICES[callee]["container"], ms))

slowest = max(((c, cal, ms) for c, v in edges.items() for cal, ms in v),
                key=lambda e: e[2])
print("slowest edge:", slowest)

## Step 3 — carry the spans with agents plus a gateway

Each container gets a lightweight agent beside it that collects local spans and forwards them; one central gateway receives from all agents and applies policy (sampling, filtering, retries) before exporting to the backend. Agents handle locality, gateways handle policy (https://opentelemetry.io/docs/collector/architecture/). One detail that matters when writing the config: a processor referenced by two pipelines gets an independent instance per pipeline, so per-signal pipelines stay isolated — and a blocked processor blocks its receiver, so heavy work belongs downstream of the gateway.

In [ ]:
config = {
    "receivers": {"otlp": {"protocols": ["grpc", "http"]}},
    "processors": {"batch": {}, "tail_sampling": {"policy": "slow-tail"}},
    "exporters": {"backend": {"endpoint": "traces-backend:4317"}},
    "service": {
        "pipelines": {
            "traces/agent": {"receivers": ["otlp"],
                                "processors": ["batch"],
                                "exporters": ["gateway"]},
            "traces/gateway": {"receivers": ["gateway"],
                                  "processors": ["tail_sampling", "batch"],
                                  "exporters": ["backend"]},
        }
    },
}
# agents export to a shared in-pipeline forwarder the gateway receives from
config["exporters"]["gateway"] = {"send_to": "gateway-receiver"}
config["service"]["pipelines"]["traces/gateway"]["receivers"] = ["gateway"]

defined = (set(config["receivers"]) | set(config["processors"])
           | set(config["exporters"]))
for name, pipe in config["service"]["pipelines"].items():
    for kind in ("receivers", "processors", "exporters"):
        for comp in pipe[kind]:
            assert comp in defined, (name, kind, comp)
print("pipelines reference only defined components: OK")
print(json.dumps(config["service"]["pipelines"], indent=2))

## Verify

The trace forms one chain frontend -> api -> db, the slowest edge ends at db, and every pipeline component in the sketch is defined. What I would try next: run the same graph builder against real span JSON exported from a compose stack, and add a metrics pipeline next to the traces one.

In [ ]:
chain = sorted(spans, key=lambda s: s["start_ms"])
assert [s["service"] for s in chain] == ["frontend", "api", "db"]
assert all(s["trace_id"] == chain[0]["trace_id"] for s in chain)
assert slowest[1] == "db", slowest
print("verify: chain %s, slowest edge ends at db — OK"
      % " -> ".join(s["service"] for s in chain))